[Mohit Saharan](https://linkedin.com/in/msaharan), 20260504, P16

Apache 2.0 License (see github.com/msaharan/dsaiengineering/LICENSE)

# TabPFN and TabICL embeddings for fraud-detection workflows - v2

This notebook treats classical supervised ML as the production workflow and tests whether TabPFN and TabICL row embeddings improve that workflow. The comparison is not TabPFN/TabICL as slow direct scorers versus XGBoost; it is raw-feature classical ML versus calibrated classical ML enhanced with offline TFM representations.

## How to run this notebook on Kaggle

1. Import this notebook into Kaggle.
2. Enable GPU acceleration.
3. Turn on Internet access because the notebook downloads packages, cuDF, model checkpoints, and the public fraud CSV.
4. Add a Kaggle secret named `TABPFN_TOKEN` if you want to run local TabPFN weights in a headless Kaggle session. You need to accept the Prior Labs license first.
5. Optionally add `HF_TOKEN` for Hugging Face downloads. TabICL is usually downloadable without it, but using a token can avoid rate-limit issues.

This notebook assumes a CUDA GPU runtime. The setup cell installs `cudf-cu12` from NVIDIA's package index, and the imports cell enables the cuDF pandas accelerator before importing pandas. TabPFN and TabICL are used as embedding generators on CUDA devices; the deployed downstream models remain classical supervised classifiers.

If a TFM embedding path fails to import or download, the notebook records the error and continues with the available feature sets.

In [ ]:
%%time
# Kaggle / Colab setup.
# Run this cell once. If imports still fail, restart the notebook session and continue below.

import subprocess
import sys

!pip uv install xgboost rich openml tabpfn tabicl

subprocess.run([sys.executable, "-m", "pip", "install", "--extra-index-url", "https://pypi.nvidia.com", "cudf-cu12"], check=True)


## 0. Imports, Secrets, and Configuration

In [ ]:
import os
import time
import warnings
from traceback import format_exception_only

# Importing cuDF and enabling the pandas accelerator. This allows us to use cuDF's GPU-accelerated DataFrame operations while maintaining compatibility with pandas code. 
import cudf.pandas
cudf.pandas.install()

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from IPython.display import display
from rich.console import Console

from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.compose import make_column_selector, make_column_transformer
from sklearn.datasets import fetch_openml
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    log_loss,
    precision_recall_curve,
    roc_auc_score,
)
from sklearn.model_selection import RandomizedSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OrdinalEncoder, StandardScaler

from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
console = Console()
console.print("[green]cuDF pandas accelerator enabled.[/green]")

SEED = 42
# Number of ensemble members for TabPFN and TabICL. More members will increase performance but also runtime.
N_TFM_ESTIMATORS = 8
# Number of tuning iterations for each classical model. More iterations will increase performance but also runtime.
CLASSICAL_TUNING_ITERATIONS = 16
# Number of parallel tuning jobs for each classical model. More jobs will increase performance but also runtime and memory usage.
CLASSICAL_TUNING_JOBS = 1
# Whether to tune the classical models. Tuning can be time-consuming, so you may want to set this to False for a quick evaluation or if you have limited computational resources.
TUNE_CLASSICAL_MODELS = True
# Fractions of the data to use for representation context, training, validation, calibration, and testing based on time. The data will be split chronologically, so setting these fractions controls which history is used to condition TFM embeddings, which later history trains the downstream classical models, and which final rows remain held out.
EMBEDDING_CONTEXT_END_FRACTION_BY_TIME = 0.20
TRAIN_END_FRACTION_BY_TIME = 0.60
VALIDATION_END_FRACTION_BY_TIME = 0.70
CALIBRATION_END_FRACTION_BY_TIME = 0.80
# Maximum number of samples to use for training, validation, and testing the classical models. Setting these limits can help reduce runtime and memory usage, especially for large datasets. The data will be sampled randomly within each split, so you will get a representative subset of the data for each phase. Adjust these limits based on the size of your dataset and the computational resources available.
MAX_TRAIN_NORMAL = 4000
MAX_CONTEXT_NORMAL = 4000
MAX_TEST_NORMAL = 20000
# Whether to include time as a feature for the models. Including time can help the models capture temporal patterns in the data, but it may also introduce noise if time is not a relevant factor for the prediction task. Consider the nature of your data and the importance of temporal information when deciding whether to use time as a feature.
USE_TIME_AS_FEATURE = False
# Whether to build the full-holdout embedding feature matrices. Keep this True for the deployment-facing comparison.
BUILD_FULL_HOLDOUT_EMBEDDINGS = True
# Number of rows per embedding extraction chunk. Larger chunks can be faster but need more GPU memory.
TFM_EMBEDDING_CHUNK_SIZE = 4096

np.random.seed(SEED)

# GPU-only device configuration for TabPFN and TabICL.
CUDA_DEVICE_COUNT = torch.cuda.device_count()
if CUDA_DEVICE_COUNT < 1:
    raise RuntimeError("This notebook requires a CUDA GPU runtime.")

TABICL_DEVICE = "cuda:0"
TABPFN_DEVICE = [f"cuda:{i}" for i in range(CUDA_DEVICE_COUNT)]

print(f"CUDA device count: {CUDA_DEVICE_COUNT}")
print(f"TabICL device: {TABICL_DEVICE}")
print(f"TabPFN device: {TABPFN_DEVICE}")
print(f"TFM ensemble members: {N_TFM_ESTIMATORS}")
print(f"Tune classical models: {TUNE_CLASSICAL_MODELS}")
print(f"Classical tuning iterations per model: {CLASSICAL_TUNING_ITERATIONS}")
print(f"Embedding context/train/validation/calibration/test time fractions: 0-{EMBEDDING_CONTEXT_END_FRACTION_BY_TIME:.0%}, {EMBEDDING_CONTEXT_END_FRACTION_BY_TIME:.0%}-{TRAIN_END_FRACTION_BY_TIME:.0%}, {TRAIN_END_FRACTION_BY_TIME:.0%}-{VALIDATION_END_FRACTION_BY_TIME:.0%}, {VALIDATION_END_FRACTION_BY_TIME:.0%}-{CALIBRATION_END_FRACTION_BY_TIME:.0%}, {CALIBRATION_END_FRACTION_BY_TIME:.0%}-100%")
print(f"Use Time as model feature: {USE_TIME_AS_FEATURE}")
print(f"Build full-holdout embedding features: {BUILD_FULL_HOLDOUT_EMBEDDINGS}")

In [ ]:
def load_secret(name):
    value = os.environ.get(name)
    if value:
        return value

    try:
        from kaggle_secrets import UserSecretsClient

        value = UserSecretsClient().get_secret(name)
        if value:
            os.environ[name] = value
            return value
    except Exception:
        pass

    try:
        from google.colab import userdata

        value = userdata.get(name)
        if value:
            os.environ[name] = value
            return value
    except Exception:
        pass

    return None


hf_token = load_secret("HF_TOKEN")
tabpfn_token = load_secret("TABPFN_TOKEN")

if hf_token:
    print("HF_TOKEN found.")
else:
    print("HF_TOKEN not found. Hugging Face downloads will use anonymous requests.")

if tabpfn_token:
    print("TABPFN_TOKEN found.")
else:
    os.environ["TABPFN_NO_BROWSER"] = "1"
    print("TABPFN_TOKEN not found. TabPFN will be skipped or fail quickly instead of waiting for browser auth.")

_ = os.environ.setdefault("TABPFN_DISABLE_TELEMETRY", "1")

## 1. Load the Credit-Card Fraud Dataset

This notebook uses the public credit-card fraud CSV originally released with anonymized transaction features, `Time`, `Amount`, and a binary fraud label. The primary loader uses a public CSV URL that preserves the `Time` column. OpenML is kept only as a fallback, and the notebook stops if the loaded data does not contain `Time`, because the workflow depends on a real temporal holdout.


In [ ]:
CREDIT_CARD_FRAUD_CSV_URL = "https://storage.googleapis.com/download.tensorflow.org/data/creditcard.csv"


def load_creditcard_fraud_data():
    try:
        df = pd.read_csv(CREDIT_CARD_FRAUD_CSV_URL)
        return df, "Class", CREDIT_CARD_FRAUD_CSV_URL
    except Exception as csv_exc:
        print(f"Primary CSV load failed: {type(csv_exc).__name__}: {csv_exc}")
        print("Falling back to OpenML data_id=1597.")

    raw = fetch_openml(data_id=1597, as_frame=True, parser="auto")
    df = raw.frame.copy()
    target_column = "Class" if "Class" in df.columns else raw.target_names[0]
    return df, target_column, "OpenML data_id=1597 fallback"


df, target_column, data_source = load_creditcard_fraud_data()

for column in df.columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

df = df.dropna(subset=[target_column]).copy()
df[target_column] = df[target_column].astype(int)

if "Time" not in df.columns:
    raise ValueError(
        "The loaded dataset does not include a `Time` column, so this notebook cannot "
        "run the temporal holdout as written. Use the CSV source that preserves `Time`, "
        "or revise the workflow to describe an ordered holdout instead."
    )

df = df.sort_values("Time").reset_index(drop=True)

print(f"Dataset source: {data_source}")
print(f"Dataset shape: {df.shape}")
print(f"Target column: {target_column}")
print(f"Time range: {df['Time'].min():.0f} to {df['Time'].max():.0f}")
display(df.head())
display(df[target_column].value_counts().rename("count").to_frame())
print(f"Fraud rate: {df[target_column].mean():.4%}")


## 2. Lightweight Data Checks

Before modeling, inspect the target imbalance, missing values, feature types, and feature cardinality.

In [ ]:
X_full = df.drop(columns=[target_column])
y_full = df[target_column].astype(int)

checks = pd.DataFrame(
    {
        "dtype": X_full.dtypes.astype(str),
        "missing_rate": X_full.isna().mean(),
        "n_unique": X_full.nunique(dropna=False),
    }
).sort_values(["missing_rate", "n_unique"], ascending=False)

summary = pd.DataFrame(
    {
        "value": [
            len(df),
            X_full.shape[1],
            int(y_full.sum()),
            y_full.mean(),
        ]
    },
    index=["rows", "features", "fraud_rows", "fraud_rate"],
)

display(summary)
display(checks.head(20))

## 3. Time-Aware Split and Representation Context

The workflow uses five chronological windows:

- earliest 20% of transactions: TFM representation context only;
- next 40% of transactions: downstream classical training window;
- next 10% of transactions: full-prevalence validation window for classical model selection;
- next 10% of transactions: full-prevalence calibration window for post-hoc probability calibration;
- final 20% of transactions: future holdout.

This split makes the embedding experiment cleaner without starving the incumbent classical workflow. TabPFN and TabICL see only an earlier labeled context when generating row embeddings for later training, validation, calibration, and holdout rows. That avoids using a row's own label inside its TFM-derived features.

The downstream production workflow remains classical ML. Logistic Regression and XGBoost are tuned on a time-aware validation fold, then evaluated with and without offline TFM embedding features. Calibration is kept because fraud teams often need more than ranking: alert thresholds, review queues, monitoring, and policy decisions usually depend on probability behavior as well.

The representation-context rows are not included in the downstream training matrices because their own labels condition the TFM context. A stronger future extension would add a separate raw-feature incumbent trained on all pre-holdout history, but the fair raw-vs-embedding comparison here keeps the downstream rows identical across feature sets.

In [ ]:
embedding_context_end_idx = int(len(df) * EMBEDDING_CONTEXT_END_FRACTION_BY_TIME)
train_end_idx = int(len(df) * TRAIN_END_FRACTION_BY_TIME)
validation_end_idx = int(len(df) * VALIDATION_END_FRACTION_BY_TIME)
calibration_end_idx = int(len(df) * CALIBRATION_END_FRACTION_BY_TIME)

embedding_context_period = df.iloc[:embedding_context_end_idx].copy()
train_period = df.iloc[embedding_context_end_idx:train_end_idx].copy()
validation_period = df.iloc[train_end_idx:validation_end_idx].copy()
calibration_period = df.iloc[validation_end_idx:calibration_end_idx].copy()
test_period = df.iloc[calibration_end_idx:].copy()


def sample_binary_window(frame, max_normal_rows, random_state=SEED):
    fraud_rows = frame[frame[target_column] == 1]
    normal_rows = frame[frame[target_column] == 0]

    if max_normal_rows is None or len(normal_rows) <= max_normal_rows:
        sampled_normal = normal_rows
    else:
        sampled_normal = normal_rows.sample(n=max_normal_rows, random_state=random_state)

    return (
        pd.concat([fraud_rows, sampled_normal], axis=0)
        .sort_values("Time")
        .reset_index(drop=True)
    )


context_df = sample_binary_window(embedding_context_period, MAX_CONTEXT_NORMAL)
train_df = sample_binary_window(train_period, MAX_TRAIN_NORMAL)
validation_df = validation_period.reset_index(drop=True)
calibration_df = calibration_period.reset_index(drop=True)
test_sample_df = sample_binary_window(test_period, MAX_TEST_NORMAL)
test_full_df = test_period.reset_index(drop=True)

feature_columns = [column for column in df.columns if column != target_column]
if not USE_TIME_AS_FEATURE and "Time" in feature_columns:
    feature_columns.remove("Time")

X_context = context_df[feature_columns].copy()
y_context = context_df[target_column].astype(int).copy()
X_train = train_df[feature_columns].copy()
y_train = train_df[target_column].astype(int).copy()
X_validation = validation_df[feature_columns].copy()
y_validation = validation_df[target_column].astype(int).copy()
X_calibration = calibration_df[feature_columns].copy()
y_calibration = calibration_df[target_column].astype(int).copy()
X_test_sample = test_sample_df[feature_columns].copy()
y_test_sample = test_sample_df[target_column].astype(int).copy()
X_test_full = test_full_df[feature_columns].copy()
y_test_full = test_full_df[target_column].astype(int).copy()

X_classical_tune = pd.concat([X_train, X_validation], axis=0).reset_index(drop=True)
y_classical_tune = pd.concat([y_train, y_validation], axis=0).reset_index(drop=True)
validation_fold_indices = np.concatenate([
    np.full(len(X_train), -1, dtype=int),
    np.zeros(len(X_validation), dtype=int),
])

classical_base_train_df = pd.concat([train_period, validation_period], axis=0).reset_index(drop=True)
classical_final_train_df = pd.concat([train_period, validation_period, calibration_period], axis=0).reset_index(drop=True)
X_classical_base_train = classical_base_train_df[feature_columns].copy()
y_classical_base_train = classical_base_train_df[target_column].astype(int).copy()
X_classical_final_train = classical_final_train_df[feature_columns].copy()
y_classical_final_train = classical_final_train_df[target_column].astype(int).copy()

split_summary = pd.DataFrame(
    [
        {
            "split": "tfm_embedding_context_sampled",
            "rows": len(context_df),
            "fraud_rows": int(y_context.sum()),
            "fraud_rate": y_context.mean(),
            "time_min": context_df["Time"].min(),
            "time_max": context_df["Time"].max(),
        },
        {
            "split": "classical_train_window_sampled",
            "rows": len(train_df),
            "fraud_rows": int(y_train.sum()),
            "fraud_rate": y_train.mean(),
            "time_min": train_df["Time"].min(),
            "time_max": train_df["Time"].max(),
        },
        {
            "split": "validation_window_full",
            "rows": len(validation_df),
            "fraud_rows": int(y_validation.sum()),
            "fraud_rate": y_validation.mean(),
            "time_min": validation_df["Time"].min(),
            "time_max": validation_df["Time"].max(),
        },
        {
            "split": "calibration_window_full",
            "rows": len(calibration_df),
            "fraud_rows": int(y_calibration.sum()),
            "fraud_rate": y_calibration.mean(),
            "time_min": calibration_df["Time"].min(),
            "time_max": calibration_df["Time"].max(),
        },
        {
            "split": "test_holdout_sampled",
            "rows": len(test_sample_df),
            "fraud_rows": int(y_test_sample.sum()),
            "fraud_rate": y_test_sample.mean(),
            "time_min": test_sample_df["Time"].min(),
            "time_max": test_sample_df["Time"].max(),
        },
        {
            "split": "test_holdout_full",
            "rows": len(test_full_df),
            "fraud_rows": int(y_test_full.sum()),
            "fraud_rate": y_test_full.mean(),
            "time_min": test_full_df["Time"].min(),
            "time_max": test_full_df["Time"].max(),
        },
    ]
)

display(split_summary)
print(f"Using Time as feature: {USE_TIME_AS_FEATURE}")
print(f"Feature count before embeddings: {len(feature_columns)}")
print(f"TFM embedding context rows: {len(X_context):,}")
print(f"Classical tuning rows: {len(X_classical_tune):,}")
print(f"Classical base-fit rows for calibration: {len(X_classical_base_train):,}")
print(f"Classical final-training rows: {len(X_classical_final_train):,}")
print(f"Classical calibration rows: {len(X_calibration):,}")


## 4. Build TFM Embedding Feature Sets

This section turns TabPFN and TabICL into offline feature generators. Each model is fitted only on the earlier representation-context window. The notebook then extracts embeddings for the later classical training, validation, calibration, and holdout windows and appends those vectors to the raw transaction features.

The downstream classifiers do not call TabPFN or TabICL at prediction time. They receive ordinary feature matrices: raw features alone, raw + TabPFN embeddings, raw + TabICL embeddings, and raw + both embedding sets when both extractors are available.

TabPFN exposes a public `get_embeddings` method. TabICL does not currently expose the same sklearn-level embedding API, so this notebook extracts row representations from the fitted TabICL model internals. For production or shared benchmark work, pin the TabICL version and treat that helper as an implementation detail to review when upgrading.

In [ ]:
def short_error(exc):
    return "".join(format_exception_only(type(exc), exc)).strip().replace("\n", " ")[:400]


def slice_rows(X, start, stop):
    if hasattr(X, "iloc"):
        return X.iloc[start:stop]
    return X[start:stop]


def to_numpy_float32(X):
    if hasattr(X, "to_numpy"):
        return X.to_numpy(dtype=np.float32)
    return np.asarray(X, dtype=np.float32)


def to_numpy_int(y):
    if hasattr(y, "to_numpy"):
        return y.to_numpy(dtype=int)
    return np.asarray(y, dtype=int)


def embedding_frame(array, prefix):
    array = np.asarray(array, dtype=np.float32)
    columns = [f"{prefix}_{idx:03d}" for idx in range(array.shape[1])]
    return pd.DataFrame(array, columns=columns)


def concat_feature_blocks(*blocks):
    return pd.concat([block.reset_index(drop=True) for block in blocks], axis=1)


def tabicl_batch_size():
    return 4


RAW_FEATURE_MATRICES = {
    "X_train": X_train.reset_index(drop=True),
    "X_validation": X_validation.reset_index(drop=True),
    "X_calibration": X_calibration.reset_index(drop=True),
    "X_test_sample": X_test_sample.reset_index(drop=True),
    "X_test_full": X_test_full.reset_index(drop=True),
    "X_classical_tune": X_classical_tune.reset_index(drop=True),
    "X_classical_base_train": X_classical_base_train.reset_index(drop=True),
    "X_classical_final_train": X_classical_final_train.reset_index(drop=True),
}


def extract_tabpfn_embedding_frames(prefix="tabpfn_emb"):
    from tabpfn import TabPFNClassifier

    model = TabPFNClassifier(
        n_estimators=N_TFM_ESTIMATORS,
        device=TABPFN_DEVICE,
        random_state=SEED,
    )

    start = time.perf_counter()
    model.fit(to_numpy_float32(X_context), to_numpy_int(y_context))
    fit_seconds = time.perf_counter() - start

    def transform(X):
        chunks = []
        transform_start = time.perf_counter()
        for row_start in range(0, len(X), TFM_EMBEDDING_CHUNK_SIZE):
            row_stop = min(row_start + TFM_EMBEDDING_CHUNK_SIZE, len(X))
            X_chunk = to_numpy_float32(slice_rows(X, row_start, row_stop))
            chunk_embeddings = model.get_embeddings(X_chunk, data_source="test").mean(axis=0)
            chunks.append(chunk_embeddings)
        return embedding_frame(np.vstack(chunks), prefix), time.perf_counter() - transform_start

    frames, transform_seconds = build_embedding_frame_set(transform)
    return frames, fit_seconds + transform_seconds


def extract_tabicl_embedding_frames(prefix="tabicl_emb"):
    from tabicl import TabICLClassifier

    model = TabICLClassifier(
        n_estimators=N_TFM_ESTIMATORS,
        device=TABICL_DEVICE,
        batch_size=tabicl_batch_size(),
        random_state=SEED,
    )

    start = time.perf_counter()
    model.fit(to_numpy_float32(X_context), to_numpy_int(y_context))
    fit_seconds = time.perf_counter() - start

    def transform(X):
        transform_start = time.perf_counter()
        output_chunks = []
        for row_start in range(0, len(X), TFM_EMBEDDING_CHUNK_SIZE):
            row_stop = min(row_start + TFM_EMBEDDING_CHUNK_SIZE, len(X))
            X_chunk = to_numpy_float32(slice_rows(X, row_start, row_stop))
            X_chunk = model.X_encoder_.transform(X_chunk)
            data = model.ensemble_generator_.transform(X_chunk, mode="both")
            representation_variants = []

            for norm_method, (Xs, ys) in data.items():
                feature_shuffles = model.ensemble_generator_.feature_shuffles_[norm_method]
                batch_size = model.batch_size or Xs.shape[0]
                n_batches = int(np.ceil(Xs.shape[0] / batch_size))

                for batch_idx in range(n_batches):
                    batch_start = batch_idx * batch_size
                    batch_stop = min(batch_start + batch_size, Xs.shape[0])
                    X_batch = torch.from_numpy(Xs[batch_start:batch_stop]).float().to(model.device_)
                    y_batch = torch.from_numpy(ys[batch_start:batch_stop]).float().to(model.device_)
                    shuffle_batch = feature_shuffles[batch_start:batch_stop]

                    with torch.no_grad():
                        representations = model.model_.row_interactor(
                            model.model_.col_embedder(
                                X_batch,
                                y_train=y_batch,
                                embed_with_test=False,
                                feature_shuffles=shuffle_batch,
                                mgr_config=model.inference_config_.COL_CONFIG,
                            ),
                            mgr_config=model.inference_config_.ROW_CONFIG,
                        )
                    test_representations = representations[:, y_batch.shape[1]:, :]
                    representation_variants.append(test_representations.float().cpu().numpy())

            output_chunks.append(np.concatenate(representation_variants, axis=0).mean(axis=0))

        return embedding_frame(np.vstack(output_chunks), prefix), time.perf_counter() - transform_start

    frames, transform_seconds = build_embedding_frame_set(transform)
    return frames, fit_seconds + transform_seconds


def build_embedding_frame_set(transform):
    transform_seconds = 0.0

    final_train_frame, seconds = transform(X_classical_final_train)
    transform_seconds += seconds
    base_train_rows = len(X_classical_base_train)
    full_train_rows = len(train_period)
    validation_rows = len(X_validation)

    frames = {
        "X_classical_final_train": final_train_frame.reset_index(drop=True),
        "X_classical_base_train": final_train_frame.iloc[:base_train_rows].reset_index(drop=True),
        "X_validation": final_train_frame.iloc[full_train_rows:full_train_rows + validation_rows].reset_index(drop=True),
        "X_calibration": final_train_frame.iloc[base_train_rows:].reset_index(drop=True),
    }

    train_frame, seconds = transform(X_train)
    transform_seconds += seconds
    frames["X_train"] = train_frame.reset_index(drop=True)

    test_sample_frame, seconds = transform(X_test_sample)
    transform_seconds += seconds
    frames["X_test_sample"] = test_sample_frame.reset_index(drop=True)

    if BUILD_FULL_HOLDOUT_EMBEDDINGS:
        test_full_frame, seconds = transform(X_test_full)
        transform_seconds += seconds
        frames["X_test_full"] = test_full_frame.reset_index(drop=True)
    else:
        frames["X_test_full"] = frames["X_test_sample"].iloc[:0].copy()

    frames["X_classical_tune"] = concat_feature_blocks(frames["X_train"], frames["X_validation"])
    return frames, transform_seconds


def raw_feature_bundle():
    return {
        "feature_set": "Raw",
        "embedding_source": "none",
        "feature_prep_seconds": 0.0,
        **RAW_FEATURE_MATRICES,
    }


def enhanced_feature_bundle(feature_set, embedding_source, source_names):
    matrices = {}
    for key, raw_matrix in RAW_FEATURE_MATRICES.items():
        matrices[key] = concat_feature_blocks(
            raw_matrix,
            *[embedding_sources[source_name]["frames"][key] for source_name in source_names],
        )

    return {
        "feature_set": feature_set,
        "embedding_source": embedding_source,
        "feature_prep_seconds": sum(embedding_sources[source_name]["seconds"] for source_name in source_names),
        **matrices,
    }


embedding_sources = {}
embedding_errors = []

for source_name, extractor in [
    ("TabPFN", extract_tabpfn_embedding_frames),
    ("TabICL", extract_tabicl_embedding_frames),
]:
    console.rule(f"{source_name} embeddings")
    try:
        frames, seconds = extractor()
        embedding_sources[source_name] = {"frames": frames, "seconds": seconds}
        print(f"Built {source_name} embedding feature matrices in {seconds:.1f} seconds.")
    except Exception as exc:
        error = short_error(exc)
        embedding_errors.append({"Embedding Source": source_name, "Error": error})
        print(f"{source_name} embedding extraction failed: {error}")

feature_bundles = [raw_feature_bundle()]
if "TabPFN" in embedding_sources:
    feature_bundles.append(enhanced_feature_bundle("Raw + TabPFN embeddings", "TabPFN", ["TabPFN"]))
if "TabICL" in embedding_sources:
    feature_bundles.append(enhanced_feature_bundle("Raw + TabICL embeddings", "TabICL", ["TabICL"]))
if {"TabPFN", "TabICL"}.issubset(embedding_sources):
    feature_bundles.append(enhanced_feature_bundle("Raw + TabPFN + TabICL embeddings", "TabPFN+TabICL", ["TabPFN", "TabICL"]))

feature_bundle_summary = pd.DataFrame(
    [
        {
            "Feature Set": bundle["feature_set"],
            "Embedding Source": bundle["embedding_source"],
            "Feature Count": bundle["X_train"].shape[1],
            "Feature Prep Seconds": bundle["feature_prep_seconds"],
            "Classical Tune Rows": len(bundle["X_classical_tune"]),
            "Classical Final Train Rows": len(bundle["X_classical_final_train"]),
        }
        for bundle in feature_bundles
    ]
)

display(feature_bundle_summary.round(2))
if embedding_errors:
    display(pd.DataFrame(embedding_errors))


## 5. Classical Model Registry and Calibration Workflow

The model registry now contains only classical supervised models. TabPFN and TabICL are no longer evaluated as direct fraud scorers; they appear only through the feature bundles built above.

For each available feature set, the notebook tunes Logistic Regression and XGBoost on a time-aware train/validation split. It then evaluates both uncalibrated and sigmoid-calibrated variants. Calibration is retained because fraud workflows often need usable probabilities for alert thresholds and operating policies, not only good ranking.

In [ ]:
def make_preprocessor():
    return make_column_transformer(
        (
            OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1),
            make_column_selector(dtype_include=["object", "category"]),
        ),
        remainder="passthrough",
    )


def make_classical_search(name, estimator, param_distributions):
    if not TUNE_CLASSICAL_MODELS or not param_distributions:
        return estimator

    cv_split = [(np.where(validation_fold_indices == -1)[0], np.where(validation_fold_indices == 0)[0])]
    return RandomizedSearchCV(
        estimator=estimator,
        param_distributions=param_distributions,
        n_iter=CLASSICAL_TUNING_ITERATIONS,
        scoring="average_precision",
        cv=cv_split,
        refit=False,
        random_state=SEED,
        n_jobs=CLASSICAL_TUNING_JOBS,
        error_score=np.nan,
        verbose=1,
    )


def classical_estimators(y_train_for_weights):
    positive_count = max(int(np.sum(y_train_for_weights == 1)), 1)
    negative_count = max(int(np.sum(y_train_for_weights == 0)), 1)
    scale_pos_weight = negative_count / positive_count
    sqrt_scale_pos_weight = np.sqrt(scale_pos_weight)

    logistic_regression = make_pipeline(
        make_preprocessor(),
        StandardScaler(),
        LogisticRegression(max_iter=3000, solver="liblinear", random_state=SEED),
    )
    xgboost = make_pipeline(
        make_preprocessor(),
        XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
            random_state=SEED,
            n_jobs=-1,
        ),
    )

    return [
        (
            "LogisticRegression",
            logistic_regression,
            {
                "logisticregression__C": [0.03, 0.1, 0.3, 1.0, 3.0, 10.0],
                "logisticregression__penalty": ["l1", "l2"],
                "logisticregression__class_weight": [None, "balanced"],
            },
        ),
        (
            "XGBoost",
            xgboost,
            {
                "xgbclassifier__n_estimators": [200, 400, 700],
                "xgbclassifier__max_depth": [2, 3, 4, 6],
                "xgbclassifier__learning_rate": [0.01, 0.03, 0.1],
                "xgbclassifier__subsample": [0.8, 1.0],
                "xgbclassifier__colsample_bytree": [0.8, 1.0],
                "xgbclassifier__scale_pos_weight": [1.0, sqrt_scale_pos_weight, scale_pos_weight],
                "xgbclassifier__reg_lambda": [1.0, 3.0, 10.0],
            },
        ),
    ]


def classification_model_list(feature_bundles, y_train_for_weights):
    models = []
    raw_bundle = feature_bundles[0]
    models.append(
        {
            "name": "Dummy[Raw]",
            "base_model": "Dummy",
            "feature_set": raw_bundle["feature_set"],
            "embedding_source": raw_bundle["embedding_source"],
            "feature_prep_seconds": raw_bundle["feature_prep_seconds"],
            "model": DummyClassifier(strategy="prior"),
            "fit_X": raw_bundle["X_classical_final_train"],
            "fit_y": y_classical_final_train,
            "evaluate_full_holdout": True,
            "model_family": "baseline",
            "calibration": "none",
        }
    )

    for bundle in feature_bundles:
        for base_name, estimator, param_distributions in classical_estimators(y_train_for_weights):
            display_name = f"{base_name}[{bundle['feature_set']}]"
            models.append(
                {
                    "name": display_name,
                    "base_model": base_name,
                    "feature_set": bundle["feature_set"],
                    "embedding_source": bundle["embedding_source"],
                    "feature_prep_seconds": bundle["feature_prep_seconds"],
                    "model": make_classical_search(display_name, estimator, param_distributions),
                    "fit_X": bundle["X_classical_tune"],
                    "fit_y": y_classical_tune,
                    "final_refit_X": bundle["X_classical_final_train"],
                    "final_refit_y": y_classical_final_train,
                    "calibration_base_fit_X": bundle["X_classical_base_train"],
                    "calibration_base_fit_y": y_classical_base_train,
                    "calibration_X": bundle["X_calibration"],
                    "calibration_y": y_calibration,
                    "evaluate_full_holdout": True,
                    "model_family": "classical",
                    "calibrate_on_validation": True,
                    "calibration": "none",
                }
            )

    return models


## 6. Evaluation Helpers

Average Precision remains the primary model-quality metric because fraud detection is a rare-event ranking problem. The notebook also tracks Brier score, log loss, alert-budget recall, downstream fit/predict time, calibration time, embedding-preparation time, and total workflow time.

`Total Seconds` is the downstream classical model time. `Workflow Seconds` adds the shared offline embedding-preparation time for feature sets that use TabPFN and/or TabICL embeddings.

In [ ]:
def row_slice(X, start, stop):
    if hasattr(X, "iloc"):
        return X.iloc[start:stop]
    return X[start:stop]


def positive_class_proba(estimator, X):
    proba = estimator.predict_proba(X)
    if proba.ndim != 2 or proba.shape[1] < 2:
        raise ValueError("Expected a two-column probability array for binary classification.")

    classes = getattr(estimator, "classes_", None)
    if classes is not None and 1 in list(classes):
        positive_index = list(classes).index(1)
    else:
        positive_index = 1

    return proba[:, positive_index]


def predict_proba_in_chunks(estimator, X, chunk_size=4096):
    chunks = []
    for start in range(0, len(X), chunk_size):
        stop = min(start + chunk_size, len(X))
        chunks.append(positive_class_proba(estimator, row_slice(X, start, stop)))
    return np.concatenate(chunks)


def top_rate_precision_recall(y_true, y_proba, rate):
    y_true = np.asarray(y_true).astype(int)
    y_proba = np.asarray(y_proba)
    order = np.argsort(-y_proba)
    k = max(1, int(np.ceil(rate * len(y_true))))
    selected = order[:k]
    positives_found = int(y_true[selected].sum())
    total_positives = max(int(y_true.sum()), 1)
    return positives_found / k, positives_found / total_positives, k


def metric_row(
    model_name,
    base_model,
    model_family,
    feature_set,
    embedding_source,
    calibration,
    holdout_name,
    y_true,
    y_proba,
    fit_seconds,
    predict_seconds,
    cv_average_precision,
    best_params,
    feature_prep_seconds=0.0,
    calibration_seconds=0.0,
    timing_notes="",
):
    y_proba = np.clip(y_proba, 1e-7, 1 - 1e-7)
    top_005_precision, top_005_recall, top_005_alerts = top_rate_precision_recall(y_true, y_proba, 0.005)
    top_010_precision, top_010_recall, top_010_alerts = top_rate_precision_recall(y_true, y_proba, 0.01)
    total_seconds = fit_seconds + calibration_seconds + predict_seconds

    return {
        "Model": model_name,
        "Base Model": base_model,
        "Family": model_family,
        "Feature Set": feature_set,
        "Embedding Source": embedding_source,
        "Calibration": calibration,
        "Holdout": holdout_name,
        "Rows": len(y_true),
        "Fraud Rate": float(np.mean(y_true)),
        "Average Precision": average_precision_score(y_true, y_proba),
        "Top 0.5% Alerts": top_005_alerts,
        "Top 0.5% Precision": top_005_precision,
        "Top 0.5% Recall": top_005_recall,
        "Top 1% Alerts": top_010_alerts,
        "Top 1% Precision": top_010_precision,
        "Top 1% Recall": top_010_recall,
        "CV/Validation Average Precision": cv_average_precision,
        "ROC AUC": roc_auc_score(y_true, y_proba),
        "Log Loss": log_loss(y_true, y_proba, labels=[0, 1]),
        "Brier Score": brier_score_loss(y_true, y_proba),
        "Balanced Accuracy @ 0.5": balanced_accuracy_score(y_true, y_proba >= 0.5),
        "Feature Prep Seconds": feature_prep_seconds,
        "Fit Seconds": fit_seconds,
        "Calibration Seconds": calibration_seconds,
        "Predict Seconds": predict_seconds,
        "Total Seconds": total_seconds,
        "Workflow Seconds": feature_prep_seconds + total_seconds,
        "Timing Notes": timing_notes,
        "Best Params": best_params,
        "Error": "",
    }


def best_estimator_from_search(search_model):
    if hasattr(search_model, "best_params_"):
        estimator = clone(search_model.estimator)
        estimator.set_params(**search_model.best_params_)
        return estimator
    return clone(search_model)


def make_prefit_calibrator(prefit_estimator, X_calibration, y_calibration, method="sigmoid"):
    try:
        from sklearn.frozen import FrozenEstimator

        calibrator = CalibratedClassifierCV(FrozenEstimator(prefit_estimator), method=method)
        return calibrator.fit(X_calibration, y_calibration)
    except Exception:
        calibrator = CalibratedClassifierCV(prefit_estimator, method=method, cv="prefit")
        return calibrator.fit(X_calibration, y_calibration)


def make_validation_calibrated_variant(search_model, spec):
    if spec.get("calibration_X") is None:
        return None, np.nan

    calibration_start = time.perf_counter()
    base_estimator = best_estimator_from_search(search_model)
    base_estimator.fit(spec["calibration_base_fit_X"], spec["calibration_base_fit_y"])
    calibrated_model = make_prefit_calibrator(base_estimator, spec["calibration_X"], spec["calibration_y"], method="sigmoid")
    calibration_seconds = time.perf_counter() - calibration_start
    return calibrated_model, calibration_seconds


def append_prediction_rows(
    rows,
    predictions,
    model_name,
    base_model,
    model_family,
    feature_set,
    embedding_source,
    calibration,
    model,
    holdouts,
    evaluate_full_holdout,
    fit_seconds,
    cv_average_precision,
    best_params,
    feature_prep_seconds=0.0,
    calibration_seconds=0.0,
    timing_notes="",
):
    for holdout_name, X_eval, y_eval in holdouts:
        if holdout_name == "full" and not evaluate_full_holdout:
            continue

        predict_start = time.perf_counter()
        y_proba = predict_proba_in_chunks(model, X_eval)
        predict_seconds = time.perf_counter() - predict_start
        y_proba = np.clip(y_proba, 1e-7, 1 - 1e-7)
        predictions[(model_name, holdout_name)] = y_proba
        rows.append(
            metric_row(
                model_name,
                base_model,
                model_family,
                feature_set,
                embedding_source,
                calibration,
                holdout_name,
                y_eval,
                y_proba,
                fit_seconds,
                predict_seconds,
                cv_average_precision,
                best_params,
                feature_prep_seconds=feature_prep_seconds,
                calibration_seconds=calibration_seconds,
                timing_notes=timing_notes,
            )
        )


def evaluate_models(model_specs, holdouts):
    rows = []
    predictions = {}

    for spec in model_specs:
        name = spec["name"]
        model = spec["model"]
        base_model = spec["base_model"]
        model_family = spec["model_family"]
        feature_set = spec["feature_set"]
        embedding_source = spec["embedding_source"]
        feature_prep_seconds = spec.get("feature_prep_seconds", 0.0)
        console.rule(name)

        try:
            fit_start = time.perf_counter()
            model.fit(spec["fit_X"], spec["fit_y"])
            search_or_fit_seconds = time.perf_counter() - fit_start

            cv_average_precision = getattr(model, "best_score_", np.nan)
            best_params = getattr(model, "best_params_", "")
            if hasattr(model, "best_score_"):
                print(f"Best validation Average Precision: {model.best_score_:.4f}")
                print(f"Best params: {model.best_params_}")

            prediction_model = model
            final_refit_seconds = 0.0
            if "final_refit_X" in spec:
                final_refit_start = time.perf_counter()
                prediction_model = best_estimator_from_search(model)
                prediction_model.fit(spec["final_refit_X"], spec["final_refit_y"])
                final_refit_seconds = time.perf_counter() - final_refit_start
                print(f"Final refit on train+validation+calibration data: {final_refit_seconds:.1f} seconds")

            fit_seconds = search_or_fit_seconds + final_refit_seconds

            if model_family == "classical":
                timing_notes = "validation-window tuning + final pre-holdout refit"
            elif model_family == "baseline":
                timing_notes = "simple baseline fit on pre-holdout data"
            else:
                timing_notes = ""

            if feature_prep_seconds > 0:
                timing_notes = f"{timing_notes}; shared offline embedding feature prep"

            append_prediction_rows(
                rows,
                predictions,
                name,
                base_model,
                model_family,
                feature_set,
                embedding_source,
                "none",
                prediction_model,
                holdouts,
                spec["evaluate_full_holdout"],
                fit_seconds,
                cv_average_precision,
                best_params,
                feature_prep_seconds=feature_prep_seconds,
                timing_notes=timing_notes,
            )

            if spec.get("calibrate_on_validation", False):
                calibrated_model, calibration_seconds = make_validation_calibrated_variant(model, spec)
                if calibrated_model is not None:
                    calibrated_name = f"{name} Calibrated"
                    append_prediction_rows(
                        rows,
                        predictions,
                        calibrated_name,
                        base_model,
                        f"{model_family}_calibrated",
                        feature_set,
                        embedding_source,
                        "sigmoid",
                        calibrated_model,
                        holdouts,
                        spec["evaluate_full_holdout"],
                        search_or_fit_seconds,
                        cv_average_precision,
                        best_params,
                        feature_prep_seconds=feature_prep_seconds,
                        calibration_seconds=calibration_seconds,
                        timing_notes="validation tuning + train/validation base fit + separate calibration-window sigmoid calibration; shared offline embedding feature prep" if feature_prep_seconds > 0 else "validation tuning + train/validation base fit + separate calibration-window sigmoid calibration",
                    )
                    print(f"Added calibrated variant: {calibrated_name} ({calibration_seconds:.1f} calibration seconds)")

            print(f"Finished {name}: fit/search/refit {fit_seconds:.1f} seconds")
        except Exception as exc:
            error = short_error(exc)
            rows.append(
                {
                    "Model": name,
                    "Base Model": base_model,
                    "Family": model_family,
                    "Feature Set": feature_set,
                    "Embedding Source": embedding_source,
                    "Calibration": spec.get("calibration", "none"),
                    "Holdout": "fit_or_predict_failed",
                    "Rows": np.nan,
                    "Fraud Rate": np.nan,
                    "Average Precision": np.nan,
                    "Top 0.5% Alerts": np.nan,
                    "Top 0.5% Precision": np.nan,
                    "Top 0.5% Recall": np.nan,
                    "Top 1% Alerts": np.nan,
                    "Top 1% Precision": np.nan,
                    "Top 1% Recall": np.nan,
                    "CV/Validation Average Precision": np.nan,
                    "ROC AUC": np.nan,
                    "Log Loss": np.nan,
                    "Brier Score": np.nan,
                    "Balanced Accuracy @ 0.5": np.nan,
                    "Feature Prep Seconds": feature_prep_seconds,
                    "Fit Seconds": np.nan,
                    "Calibration Seconds": np.nan,
                    "Predict Seconds": np.nan,
                    "Total Seconds": np.nan,
                    "Workflow Seconds": np.nan,
                    "Timing Notes": "",
                    "Best Params": "",
                    "Error": error,
                }
            )
            print(f"{name} failed: {error}")

    summary = pd.DataFrame(rows)
    return summary.sort_values(["Holdout", "Average Precision"], ascending=[True, False], na_position="last"), predictions


def plot_precision_recall_curves(predictions, y_by_holdout, holdout_name, title, output_path=None):
    fig, ax = plt.subplots(figsize=(8, 6))
    plotted = False

    for (model_name, prediction_holdout), y_proba in predictions.items():
        if prediction_holdout != holdout_name:
            continue
        y_true = y_by_holdout[holdout_name]
        precision, recall, _ = precision_recall_curve(y_true, y_proba)
        ap = average_precision_score(y_true, y_proba)
        ax.plot(recall, precision, label=f"{model_name} AP={ap:.3f}")
        plotted = True

    if not plotted:
        print(f"No successful model predictions to plot for holdout={holdout_name}.")
        return None

    positive_rate = np.mean(y_by_holdout[holdout_name])
    ax.axhline(
        positive_rate,
        color="gray",
        linestyle="--",
        linewidth=1,
        label=f"base rate={positive_rate:.3f}",
    )
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_title(title)
    ax.legend(fontsize=7)
    ax.grid(alpha=0.25)
    plt.tight_layout()

    if output_path is not None:
        fig.savefig(output_path, dpi=160, bbox_inches="tight")
        print(f"Saved {output_path}")

    return fig


## 7. Run the Raw vs Embedding-Enhanced Classical Workflow

This is the main workflow test. The baseline is a normal classical fraud model pipeline on raw transaction features. The enhancement is the same classical pipeline after appending TabPFN and/or TabICL row embeddings that were generated from an earlier historical context.

Each Logistic Regression and XGBoost feature-set variant is tuned on the train/validation split, refit on pre-holdout history, and then optionally calibrated with a separate calibration window. The full holdout keeps the original fraud base rate and is the deployment-facing view.

In [ ]:
model_specs = classification_model_list(feature_bundles, y_classical_tune)
print("Models:", [spec["name"] for spec in model_specs])

y_by_holdout = {
    "sampled": y_test_sample,
    "full": y_test_full,
}

# Each spec carries its own feature matrices, so replace the raw holdout X with the matching bundle view.
spec_holdouts = {}
for spec in model_specs:
    matching_bundle = next(bundle for bundle in feature_bundles if bundle["feature_set"] == spec["feature_set"])
    spec_holdouts[spec["name"]] = [
        ("sampled", matching_bundle["X_test_sample"], y_test_sample),
        ("full", matching_bundle["X_test_full"], y_test_full),
    ]

fraud_rows = []
fraud_predictions = {}
for spec in model_specs:
    spec_summary, spec_predictions = evaluate_models([spec], spec_holdouts[spec["name"]])
    fraud_rows.append(spec_summary)
    fraud_predictions.update(spec_predictions)

fraud_summary = pd.concat(fraud_rows, ignore_index=True).sort_values(
    ["Holdout", "Average Precision"], ascending=[True, False], na_position="last"
)

performance_columns = [
    "Model",
    "Base Model",
    "Feature Set",
    "Embedding Source",
    "Calibration",
    "Family",
    "Holdout",
    "Rows",
    "Fraud Rate",
    "Average Precision",
    "Top 0.5% Precision",
    "Top 0.5% Recall",
    "Top 1% Precision",
    "Top 1% Recall",
    "CV/Validation Average Precision",
    "ROC AUC",
    "Log Loss",
    "Brier Score",
    "Feature Prep Seconds",
    "Fit Seconds",
    "Calibration Seconds",
    "Predict Seconds",
    "Total Seconds",
    "Workflow Seconds",
]
display(fraud_summary[performance_columns].round(4))

comparison_view = (
    fraud_summary.loc[
        (fraud_summary["Base Model"].isin(["LogisticRegression", "XGBoost"]))
        & fraud_summary["Average Precision"].notna(),
        [
            "Base Model",
            "Calibration",
            "Holdout",
            "Feature Set",
            "Average Precision",
            "Top 1% Recall",
            "Brier Score",
            "Workflow Seconds",
        ],
    ]
    .pivot_table(
        index=["Base Model", "Calibration", "Holdout"],
        columns="Feature Set",
        values=["Average Precision", "Top 1% Recall", "Brier Score", "Workflow Seconds"],
        aggfunc="first",
    )
)
display(comparison_view.round(4))

best_params = fraud_summary.loc[
    (fraud_summary["Family"] == "classical")
    & (fraud_summary["Best Params"].astype(str).str.len() > 0),
    ["Base Model", "Feature Set", "CV/Validation Average Precision", "Best Params"],
].drop_duplicates(subset=["Base Model", "Feature Set"])
if len(best_params) > 0:
    display(best_params)

timing_notes = fraud_summary.loc[
    fraud_summary["Timing Notes"].astype(str).str.len() > 0,
    ["Model", "Feature Set", "Calibration", "Timing Notes"],
].drop_duplicates()
if len(timing_notes) > 0:
    display(timing_notes)

if fraud_summary["Error"].str.len().sum() > 0:
    display(fraud_summary.loc[fraud_summary["Error"].str.len() > 0, ["Model", "Feature Set", "Error"]])


### What the Holdout Results Mean

The full holdout is the deployment-facing view because it keeps the original final-window fraud base rate. The sampled holdout is mainly a faster comparison slice.

The question is now directly useful for a fraud team: if the organization keeps Logistic Regression or XGBoost as the production scorer, do TabPFN or TabICL embeddings improve ranking quality, alert-budget recall, or probability quality enough to justify the extra offline feature-generation path?

Read calibrated rows separately from uncalibrated rows. Average Precision measures ranking, while Brier score and log loss say more about whether a score can be treated as a probability.

## 8. Precision-Recall Curves

For rare-event detection, the precision-recall curve is usually more informative than the ROC curve. The dashed horizontal line is the fraud rate in the evaluation sample.

In [ ]:
_ = plot_precision_recall_curves(
    fraud_predictions,
    y_by_holdout,
    holdout_name="sampled",
    title="Credit-Card Fraud Detection - Sampled Holdout Precision-Recall Curves",
    output_path="fraud_precision_recall_curves_sampled.png",
)

_ = plot_precision_recall_curves(
    fraud_predictions,
    y_by_holdout,
    holdout_name="full",
    title="Credit-Card Fraud Detection - Full Holdout Precision-Recall Curves",
    output_path="fraud_precision_recall_curves_full.png",
)


### What the Precision-Recall Curves Show

The precision-recall curves compare classical production-style scorers under different feature sets. The important read is whether adding TabPFN and/or TabICL embeddings moves the Logistic Regression or XGBoost curve upward on the full holdout, not whether a TFM directly replaces the classical model.

The dashed baseline is near zero because fraud is rare. This is why Average Precision and alert-budget metrics are more informative than plain accuracy for this workflow.

## 9. Runtime vs Average Precision

These plots make the practical tradeoff visible: model quality versus workflow time. The dummy baseline is excluded so the real model cluster remains readable.

For raw feature models, workflow time is just downstream fit, optional calibration, and prediction. For embedding-enhanced models, workflow time also includes the shared offline TabPFN/TabICL embedding preparation cost.

In [ ]:
runtime_ap_summary = (
    fraud_summary.loc[
        (fraud_summary["Base Model"] != "Dummy")
        & fraud_summary["Average Precision"].notna()
        & fraud_summary["Workflow Seconds"].notna(),
        ["Model", "Base Model", "Feature Set", "Calibration", "Holdout", "Rows", "Average Precision", "Brier Score", "Workflow Seconds"],
    ]
    .sort_values(["Holdout", "Average Precision"], ascending=[True, False])
    .reset_index(drop=True)
)

display(runtime_ap_summary.round(4))


def plot_runtime_vs_ap(summary, holdout_name, output_path):
    plot_df = summary[summary["Holdout"] == holdout_name].copy()
    plot_df = plot_df.sort_values("Workflow Seconds").reset_index(drop=True)

    if len(plot_df) == 0:
        print(f"No successful {holdout_name}-holdout model results to plot.")
        return None

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.scatter(plot_df["Workflow Seconds"], plot_df["Average Precision"], s=90)

    for idx, row in plot_df.iterrows():
        ax.annotate(
            str(idx + 1),
            (row["Workflow Seconds"], row["Average Precision"]),
            ha="center",
            va="center",
            color="white",
            fontweight="bold",
            fontsize=8,
        )

    legend_text = "\n".join(
        f"{idx + 1}. {row['Base Model']} | {row['Feature Set']} | {row['Calibration']} | AP={row['Average Precision']:.3f} | {row['Workflow Seconds']:.1f}s"
        for idx, row in plot_df.iterrows()
    )
    ax.text(
        1.02,
        0.5,
        legend_text,
        transform=ax.transAxes,
        va="center",
        ha="left",
        fontsize=8,
        bbox={"boxstyle": "round,pad=0.4", "facecolor": "white", "edgecolor": "0.8"},
    )

    ax.set_xscale("log")
    ax.set_xlabel(f"workflow seconds on {holdout_name} holdout (log scale)")
    ax.set_ylabel("Average Precision")
    ax.set_title(f"Fraud Detection: {holdout_name.title()}-Holdout Quality vs Workflow Time")
    ax.grid(alpha=0.25)
    fig.subplots_adjust(right=0.58)
    fig.savefig(output_path, dpi=160, bbox_inches="tight")
    print(f"Saved {output_path}")
    return fig


_ = plot_runtime_vs_ap(
    runtime_ap_summary,
    holdout_name="sampled",
    output_path="fraud_runtime_vs_average_precision_sampled.png",
)

_ = plot_runtime_vs_ap(
    runtime_ap_summary,
    holdout_name="full",
    output_path="fraud_runtime_vs_average_precision_full.png",
)


### What the Runtime Comparison Means

The runtime plot should not be read as a direct scorer race between TFMs and XGBoost. It asks a different question: what happens when the production scorer remains classical but an offline TFM representation step is added upstream?

If embedding-enhanced rows improve full-holdout AP, top-alert recall, or calibration enough, the added representation step may be worth considering for batch-scored workflows. If not, the classical raw-feature workflow is the more deployable answer.

## 10. Alert-Budget and Operating-Point View

Fraud teams often cannot investigate every transaction. This section converts model scores into operating decisions: top-percentage alert budgets, fixed alert counts, and the number of alerts needed to reach target recall. This is more deployment-like than ranking models only by Average Precision.


In [ ]:
def alert_budget_table(y_true, y_proba, alert_rates=(0.005, 0.01, 0.02, 0.05)):
    y_true = np.asarray(y_true).astype(int)
    y_proba = np.asarray(y_proba)
    order = np.argsort(-y_proba)
    total_positives = max(y_true.sum(), 1)

    rows = []
    for rate in alert_rates:
        k = max(1, int(np.ceil(rate * len(y_true))))
        selected = order[:k]
        positives_found = int(y_true[selected].sum())
        rows.append(
            {
                "alert_type": "top_rate",
                "alert_value": rate,
                "alerts": k,
                "frauds_found": positives_found,
                "precision": positives_found / k,
                "recall": positives_found / total_positives,
            }
        )

    return pd.DataFrame(rows)


def fixed_alert_count_table(y_true, y_proba, alert_counts=(100, 250, 500, 1000)):
    y_true = np.asarray(y_true).astype(int)
    y_proba = np.asarray(y_proba)
    order = np.argsort(-y_proba)
    total_positives = max(y_true.sum(), 1)

    rows = []
    for k in alert_counts:
        k = min(k, len(y_true))
        selected = order[:k]
        positives_found = int(y_true[selected].sum())
        rows.append(
            {
                "alert_type": "top_count",
                "alert_value": k,
                "alerts": k,
                "frauds_found": positives_found,
                "precision": positives_found / k,
                "recall": positives_found / total_positives,
            }
        )

    return pd.DataFrame(rows)


def target_recall_table(y_true, y_proba, target_recalls=(0.80, 0.90)):
    y_true = np.asarray(y_true).astype(int)
    y_proba = np.asarray(y_proba)
    order = np.argsort(-y_proba)
    sorted_true = y_true[order]
    cumulative_frauds = np.cumsum(sorted_true)
    total_positives = max(y_true.sum(), 1)

    rows = []
    for target_recall in target_recalls:
        required_frauds = target_recall * total_positives
        k = int(np.searchsorted(cumulative_frauds, required_frauds, side="left") + 1)
        k = min(k, len(y_true))
        threshold = y_proba[order[k - 1]]
        positives_found = int(cumulative_frauds[k - 1])
        rows.append(
            {
                "target_recall": target_recall,
                "threshold": threshold,
                "alerts": k,
                "frauds_found": positives_found,
                "precision": positives_found / k,
                "recall": positives_found / total_positives,
            }
        )

    return pd.DataFrame(rows)


def all_operating_tables(predictions, y_by_holdout):
    budget_rows = []
    recall_rows = []

    for (model_name, holdout_name), y_proba in predictions.items():
        y_true = y_by_holdout[holdout_name]

        budget_table = pd.concat(
            [
                alert_budget_table(y_true, y_proba),
                fixed_alert_count_table(y_true, y_proba),
            ],
            ignore_index=True,
        )
        budget_table.insert(0, "Holdout", holdout_name)
        budget_table.insert(0, "Model", model_name)
        budget_rows.append(budget_table)

        recall_table = target_recall_table(y_true, y_proba)
        recall_table.insert(0, "Holdout", holdout_name)
        recall_table.insert(0, "Model", model_name)
        recall_rows.append(recall_table)

    return pd.concat(budget_rows, ignore_index=True), pd.concat(recall_rows, ignore_index=True)


if not fraud_predictions:
    print("No successful model predictions available for operating-point analysis.")
else:
    alert_summary, recall_summary = all_operating_tables(fraud_predictions, y_by_holdout)
    display(alert_summary.round(4))
    display(recall_summary.round(4))


### What the alert-budget tables mean

This is the most deployment-like view in the notebook. Fraud teams usually operate review queues, not abstract model scores. The target-recall table shows how many transactions would need to be reviewed to recover a chosen fraction of fraud cases.

When headline Average Precision values are close, compare the alert counts needed at the same target recall and the frauds found at the same alert budget. A model with similar AP can still be operationally better if it finds the same fraud volume with fewer review slots.

The thresholds in this table are score cutoffs, not guaranteed calibrated fraud probabilities. They should be treated as operating-point choices for ranking and queue construction unless the probability calibration has been validated separately.


## 11. Calibration Diagnostics

Fraud scores are often used for ranking, but operational thresholds depend on probability behavior. Brier score is reported in the main metric table. The calibrated classical variants use a separate full-prevalence calibration window rather than the validation window used for model selection.

The plots below are zoomed into the rare-event probability range. A full 0-1 calibration plot is not very useful here because almost every meaningful point sits near zero. With only a few dozen fraud examples in the validation and calibration windows, these curves are diagnostics for model behavior, not a production calibration certificate.

In [ ]:
def plot_calibration_curves(predictions, y_by_holdout, holdout_name, output_path=None, max_axis=0.15):
    fig, ax = plt.subplots(figsize=(7, 6))
    plotted = False

    for (model_name, prediction_holdout), y_proba in predictions.items():
        if prediction_holdout != holdout_name:
            continue
        y_true = y_by_holdout[holdout_name]
        prob_true, prob_pred = calibration_curve(
            y_true,
            y_proba,
            n_bins=10,
            strategy="quantile",
        )
        ax.plot(prob_pred, prob_true, marker="o", label=model_name)
        plotted = True

    if not plotted:
        print(f"No successful model predictions to plot for holdout={holdout_name}.")
        return None

    ax.plot([0, max_axis], [0, max_axis], color="gray", linestyle="--", linewidth=1)
    ax.set_xlim(-0.005, max_axis)
    ax.set_ylim(-0.005, max_axis)
    ax.set_xlabel("Mean predicted probability")
    ax.set_ylabel("Observed fraud rate")
    ax.set_title(f"Calibration Curves - {holdout_name.title()} Holdout (Zoomed)")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.25)
    plt.tight_layout()

    if output_path is not None:
        fig.savefig(output_path, dpi=160, bbox_inches="tight")
        print(f"Saved {output_path}")

    return fig


_ = plot_calibration_curves(
    fraud_predictions,
    y_by_holdout,
    holdout_name="sampled",
    output_path="fraud_calibration_curves_sampled.png",
)

_ = plot_calibration_curves(
    fraud_predictions,
    y_by_holdout,
    holdout_name="full",
    output_path="fraud_calibration_curves_full.png",
)

calibration_quality_view = (
    fraud_summary.loc[
        fraud_summary["Average Precision"].notna(),
        ["Model", "Family", "Holdout", "Average Precision", "Brier Score", "Log Loss"],
    ]
    .sort_values(["Holdout", "Average Precision"], ascending=[True, False])
    .reset_index(drop=True)
)

display(calibration_quality_view.round(4))


### What the Calibration Results Mean

The calibration view separates ranking quality from probability quality for the classical workflow. The embedding-enhanced variants can improve ranking while still needing calibration before their scores are treated as probabilities.

For a production-style fraud workflow, the useful read is not only which feature set has the best Average Precision. Check whether the calibrated raw or embedding-enhanced model gives a better combination of AP, Brier score, log loss, and alert-budget recall on the full holdout.

## 12. Leakage and Reuse Checklist

This dataset is anonymized, so some production checks cannot be fully resolved. The point of this section is to make the reusable workflow explicit: before trusting any fraud result, check chronological ordering, feature availability, duplicates, target leakage, and entity-level leakage.

Rows can share the same `Time` value, so the split check below is about chronological ordering by row position after sorting, not a claim that boundary timestamp values are strictly separated. Since `Time` is excluded from model features, equal boundary timestamps are a split-design caveat rather than feature leakage.

The duplicate-row check is expanded because exact duplicates and duplicate model-feature rows can change alert-budget interpretation. The stricter model-feature check ignores `Time` and `Class`, matching what the models actually see when `USE_TIME_AS_FEATURE = False`. In a real fraud project, repeated-looking transactions may be legitimate recurring payments, retries, or duplicated records. This public dataset does not expose enough raw identifiers to fully adjudicate them, so the notebook reports the issue rather than silently ignoring it.


In [ ]:
window_labels = pd.Series(index=df.index, dtype="object")
window_labels.iloc[:embedding_context_end_idx] = "embedding_context"
window_labels.iloc[embedding_context_end_idx:train_end_idx] = "train"
window_labels.iloc[train_end_idx:validation_end_idx] = "validation"
window_labels.iloc[validation_end_idx:calibration_end_idx] = "calibration"
window_labels.iloc[calibration_end_idx:] = "test"


def summarize_duplicate_groups(group_hashes, labels, class_values):
    duplicate_frame = pd.DataFrame(
        {
            "duplicate_group": group_hashes,
            "window": labels,
            "Class": class_values.astype(int),
        }
    )
    group_sizes = duplicate_frame.groupby("duplicate_group").size()
    repeated_group_ids = group_sizes[group_sizes > 1].index
    duplicate_detail = duplicate_frame[duplicate_frame["duplicate_group"].isin(repeated_group_ids)].copy()

    if len(duplicate_detail) == 0:
        duplicate_group_summary = pd.DataFrame(
            columns=["duplicate_group", "rows", "windows", "fraud_rows"]
        )
        return duplicate_detail, duplicate_group_summary, 0

    duplicate_group_summary = (
        duplicate_detail.groupby("duplicate_group")
        .agg(rows=("Class", "size"), windows=("window", "nunique"), fraud_rows=("Class", "sum"))
        .reset_index()
    )
    cross_window_groups = int((duplicate_group_summary["windows"] > 1).sum())
    return duplicate_detail, duplicate_group_summary, cross_window_groups


exact_duplicate_group = pd.util.hash_pandas_object(df, index=False)
feature_duplicate_group = pd.util.hash_pandas_object(df[feature_columns], index=False)

exact_duplicate_detail, exact_duplicate_group_summary, exact_cross_window_groups = summarize_duplicate_groups(
    exact_duplicate_group,
    window_labels,
    df[target_column],
)
feature_duplicate_detail, feature_duplicate_group_summary, feature_cross_window_groups = summarize_duplicate_groups(
    feature_duplicate_group,
    window_labels,
    df[target_column],
)

exact_duplicate_rows = int(len(exact_duplicate_detail))
feature_duplicate_rows = int(len(feature_duplicate_detail))

feature_cross_window_fraud_rows = 0
if len(feature_duplicate_group_summary) > 0:
    feature_cross_window_ids = feature_duplicate_group_summary.loc[
        feature_duplicate_group_summary["windows"] > 1,
        "duplicate_group",
    ]
    feature_cross_window_fraud_rows = int(
        feature_duplicate_detail.loc[
            feature_duplicate_detail["duplicate_group"].isin(feature_cross_window_ids),
            "Class",
        ].sum()
    )

duplicate_diagnostics = pd.DataFrame(
    [
        {"metric": "exact_duplicate_rows_including_time_and_target", "value": exact_duplicate_rows},
        {"metric": "exact_duplicate_groups_including_time_and_target", "value": len(exact_duplicate_group_summary)},
        {"metric": "exact_duplicate_groups_crossing_time_windows", "value": exact_cross_window_groups},
        {"metric": "model_feature_duplicate_rows", "value": feature_duplicate_rows},
        {"metric": "model_feature_duplicate_groups", "value": len(feature_duplicate_group_summary)},
        {"metric": "model_feature_duplicate_groups_crossing_time_windows", "value": feature_cross_window_groups},
        {"metric": "fraud_rows_inside_cross_window_model_feature_duplicate_groups", "value": feature_cross_window_fraud_rows},
    ]
)
display(duplicate_diagnostics)

if exact_duplicate_rows > 0:
    print("Exact duplicate rows by time window and class:")
    exact_duplicate_by_window_and_class = (
        exact_duplicate_detail.groupby(["window", "Class"])
        .size()
        .unstack(fill_value=0)
        .rename(columns={0: "non_fraud_duplicate_rows", 1: "fraud_duplicate_rows"})
        .reset_index()
    )
    display(exact_duplicate_by_window_and_class)
else:
    print("No exact duplicate rows found when Time and Class are included.")

if feature_duplicate_rows > 0:
    print("Largest duplicate groups based on model features only:")
    display(
        feature_duplicate_group_summary.sort_values(["windows", "rows"], ascending=[False, False])
        .head(10)
        .drop(columns=["duplicate_group"])
    )

    if feature_cross_window_groups > 0:
        print("Model-feature duplicate groups crossing time windows need review:")
        display(
            feature_duplicate_group_summary.loc[feature_duplicate_group_summary["windows"] > 1]
            .sort_values(["windows", "rows"], ascending=[False, False])
            .head(10)
            .drop(columns=["duplicate_group"])
        )
    else:
        print("No duplicate model-feature groups cross time windows.")
else:
    print("No duplicate model-feature rows found.")

chronological_split_ordered = (
    embedding_context_period["Time"].max() <= train_period["Time"].min()
    and train_period["Time"].max() <= validation_period["Time"].min()
    and validation_period["Time"].max() <= calibration_period["Time"].min()
    and calibration_period["Time"].max() <= test_period["Time"].min()
)

duplicate_status = "review" if exact_duplicate_rows > 0 or feature_cross_window_groups > 0 else "pass"

leakage_checks = pd.DataFrame(
    [
        {
            "Check": "Target excluded from features",
            "Status": "pass" if target_column not in feature_columns else "fail",
            "Evidence": f"target_column={target_column}; feature_count={len(feature_columns)}",
        },
        {
            "Check": "Time column preserved for temporal split",
            "Status": "pass" if "Time" in df.columns else "fail",
            "Evidence": f"Time range {df['Time'].min():.0f} to {df['Time'].max():.0f}" if "Time" in df.columns else "Time missing",
        },
        {
            "Check": "Time used as model feature",
            "Status": "review" if "Time" in feature_columns else "not_used_as_feature",
            "Evidence": f"USE_TIME_AS_FEATURE={USE_TIME_AS_FEATURE}; Time in feature_columns={'Time' in feature_columns}",
        },
        {
            "Check": "Chronological split order",
            "Status": "pass" if chronological_split_ordered else "review",
            "Evidence": f"sorted by Time before splitting; equal boundary timestamps can occur; context max={embedding_context_period['Time'].max():.0f}, train min={train_period['Time'].min():.0f}, train max={train_period['Time'].max():.0f}, validation min={validation_period['Time'].min():.0f}, validation max={validation_period['Time'].max():.0f}, calibration min={calibration_period['Time'].min():.0f}, calibration max={calibration_period['Time'].max():.0f}, test min={test_period['Time'].min():.0f}",
        },
        {
            "Check": "Duplicate rows and duplicate model-feature rows",
            "Status": duplicate_status,
            "Evidence": f"exact_duplicate_rows={exact_duplicate_rows:,}; exact_cross_window_groups={exact_cross_window_groups:,}; model_feature_duplicate_rows={feature_duplicate_rows:,}; model_feature_cross_window_groups={feature_cross_window_groups:,}",
        },
        {
            "Check": "Future aggregate features",
            "Status": "unknown",
            "Evidence": "V1-V28 are anonymized PCA-style features; raw feature lineage is not available in this public dataset.",
        },
        {
            "Check": "Entity-level leakage / customer grouping",
            "Status": "unknown",
            "Evidence": "No customer/card/account identifier is available, so group-based splitting cannot be tested here.",
        },
        {
            "Check": "Label delay / chargeback availability",
            "Status": "unknown",
            "Evidence": "The public dataset does not expose when the fraud label became known.",
        },
    ]
)

display(leakage_checks)


### What the leakage checklist means

The main structural checks are the ones professionals should expect: the target is excluded from features, `Time` is used for chronological splitting rather than modeling, and the split is chronologically ordered even though equal boundary timestamp values can occur.

The duplicate diagnostics report two levels: exact duplicate rows including `Time` and `Class`, and duplicate model-feature rows using only the features the models actually see. Exact duplicates can affect data-quality interpretation. Model-feature duplicates crossing time windows are the stricter leakage signal because they could make train and holdout examples indistinguishable to the model.

Important production questions remain unresolved because the public dataset is anonymized. In a real fraud workflow, I would still need customer/account identifiers for group-aware splitting, raw feature lineage to check future aggregates, and label timing to model chargeback or investigation delay.


## 13. Time Feature Policy

The default run sets `USE_TIME_AS_FEATURE = False`. `Time` is still essential for temporal splitting, but the model features exclude it so the raw and embedding-enhanced comparisons do not depend on position in this public dataset.

If you want a sensitivity check, rerun the notebook with `USE_TIME_AS_FEATURE = True` and compare the full-holdout and alert-budget tables. Treat any large gain from `Time` as a signal to investigate whether the feature would be available and stable in the intended production setting.